In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, upper, trim, when, lit, concat, year, month, quarter
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType


df = spark.table("house_prices")
df.printSchema()
df.show(5)




In [0]:
# Real world ingestion

# from pyspark.sql.functions import col, current_timestamp, lit

# # Storage account details - would come from config or secrets
# storage_account = "yourstorageaccount"
# container = "raw"
# file_path = "house_prices/2024/01/pp-monthly.csv"

# # Authenticate using service principal or managed identity
# # In real world this would use Azure Key Vault secrets
# spark.conf.set(
#     f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
#     "OAuth"
# )
# spark.conf.set(
#     f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
#     "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
# )
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
#     dbutils.secrets.get(scope="key-vault", key="service-principal-client-id")
# )
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
#     dbutils.secrets.get(scope="key-vault", key="service-principal-client-secret")
# )

# # Read from ADLS
# bronze_df = (spark.read
#     .format("csv")
#     .option("header", "false")
#     .option("inferSchema", "true")
#     .load(f"abfss://{container}@{storage_account}.dfs.core.windows.net/{file_path}")
#     .toDF(*columns)
# )

# # Add metadata columns - common in real pipelines
# bronze_df = (bronze_df
#     .withColumn("ingested_at", current_timestamp())
#     .withColumn("source_file", lit(file_path))
# )

# # Write to Delta bronze table
# (bronze_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable("bronze.house_prices")
# )

# print(f"Bronze layer: {bronze_df.count()} records loaded")

In [0]:
# from pyspark.sql.functions import col, to_timestamp, upper, trim, when, lit, year, month, quarter, concat, round
# from pyspark.sql.types import DoubleType

# Bronze - raw data
bronze_df = spark.table("house_prices")
print(f"Bronze layer: {bronze_df.count()} records loaded")

In [0]:
from pyspark.sql.functions import col, to_timestamp, upper, trim, when, lit, current_timestamp, concat_ws, count, percentile_approx, sum as spark_sum, mean as spark_mean, min as spark_min, max as spark_max

def clean(df):
    
    # add rejection_reason column defaulting to None
    df = df.withColumn('rejection_reason', lit(None).cast("string"))
    
    # parse transfer_date
    df = df.withColumn('transfer_date', to_timestamp(col('transfer_date'), "yyyy-MM-dd HH:mm"))
    
    # null checks
    df = df.withColumn('rejection_reason',
                       when(col('transaction_id').isNull() |
                                col('price').isNull() |
                            col('transfer_date').isNull() |
                            col('postcode').isNull(), 
                            "NULL value detected in key column")
                       .otherwise(col("rejection_reason")))

    # price check
    df = df.withColumn('rejection_reason',
                       when((col('price') <= 0) &
                       (col('rejection_reason').isNull()),
                       "Price is less than or equal to zero")
                       .otherwise(col("rejection_reason"))
    )

    
    # property_type standardise and validate
    df = df.withColumn('property_type', upper(col('property_type')))
    df = df.withColumn('rejection_reason',
                       when((~col('property_type').isin('D', 'S', 'T', 'F', 'O')) &
                       (col('rejection_reason').isNull()),
                       "Invalid property type")
                       .otherwise(col('rejection_reason'))
                       )
    
    # duration standardise and validate
    df = df.withColumn('duration', upper(col('duration')))
    df = df.withColumn('rejection_reason',
                       when((~col('duration').isin('F', 'L')) &
                            (col('rejection_reason').isNull()),
                            "Invalid duration")
                       .otherwise(col('rejection_reason')))
    
    # split into clean and rejected
    df_rejected = df.filter(col('rejection_reason').isNotNull())
    df_clean = df.filter(col('rejection_reason').isNull())
    df_clean = df_clean.drop('rejection_reason')
    
    # log counts
    print(f"Records passed: {df_clean.count()}, Records rejected: {df_rejected.count()}")
    
    return df_clean, df_rejected

In [0]:
def transform(df):

    df = df.withColumn('transfer_year', year(col('transfer_date')))
    df = df.withColumn('transfer_month', month(col('transfer_date')))
    df = df.withColumn('transfer_quarter', concat(lit("Q"), quarter(col('transfer_date')).cast("string")))

    df = df.withColumn('price_band',
                       when(col('price') < 200000, "LOW")
                       .when((col('price') >= 200000) & (col('price') < 500000), "MID")
                       .when((col('price') >= 500000) & (col('price') < 1000000), "HIGH")
                       .otherwise("PREMIUM")
                       )
    df = df.withColumn('is_new_build', col('old_new') == 'Y')

    df = df.withColumn('full_address', concat_ws(' ', col('paon'), col('street'), col('city'), col('postcode')))
  
    return df

In [0]:


def summarise(df):

    sum_df = df.groupBy('county', 'property_type').agg(
        count(col("transaction_id")).alias("total_transactions"),
        round(spark_sum(col('price')),2).alias("total_volume"),
        round(spark_mean(col('price')),2).alias("avg_price"),
        percentile_approx(col('price'), 0.5).alias("median_price"),
        spark_min(col('price')).alias('min_price'),
        spark_max(col('price')).alias('max_price'),
        spark_sum(when(col('old_new') == 'Y', 1).otherwise(0)).alias('new_build')
    )

    sum_df = sum_df.withColumn('new_build_pct',
                               round((col('new_build') / col('total_transactions')) * 100, 1)
                               )

    sum_df = sum_df.drop('new_build')

    return sum_df




In [0]:
def save_outputs(silver_df, gold_df, rejected_df):

    try:
        silver_df.write.format("delta").mode("overwrite").saveAsTable("silver.house_prices")
        print("silver.house_prices written successfully")
    except Exception as e:
        print(f"Failed to write silver.house_prices: {e}")

    try:
        rejected_df.write.format("delta").mode("overwrite").saveAsTable("silver.house_prices_rejected")
        print("silver.house_prices_rejected written successfully")
    except Exception as e:
        print(f"Failed to write silver.house_prices_rejected: {e}")
              
    try:
        gold_df.write.format("delta").mode("overwrite").saveAsTable("gold.house_prices_summary")
        print("gold.house_prices_summary written successfully")
    except Exception as e:
        print(f"Failed to write gold.house_prices_summary: {e}")


    # # if writing to parquet on real Azure environment
    # df.write.parquet("abfss://silver@yourstorageaccount.dfs.core.windows.net/house_prices_rejected")


    # # if writing to csv on real Azure environment
    # df.write.csv("abfss://gold@yourstorageaccount.dfs.core.windows.net/house_prices_summary", header=True)


In [0]:
# create schemas
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# run pipeline
clean_df, rejected_df = clean(bronze_df)
transformed_df = transform(clean_df)
sum_df = summarise(transformed_df)
save_outputs(transformed_df, sum_df, rejected_df)

In [0]:
# spark.sql("SELECT COUNT(*) FROM silver.house_prices").show()
# spark.sql("SELECT COUNT(*) FROM silver.house_prices_rejected").show()
# spark.sql("SELECT COUNT(*) FROM gold.house_prices_summary").show()

display(spark.sql("SELECT * FROM silver.house_prices LIMIT 10"))
display(spark.sql("SELECT * FROM silver.house_prices_rejected LIMIT 10"))
display(spark.sql("SELECT * FROM gold.house_prices_summary LIMIT 10"))